# K-Means Clustering — Risk Tier AssignmentUses age and BMI to cluster customers into risk tiers (LOW_RISK, MEDIUM_RISK, HIGH_RISK) with K-Means.  K is chosen automatically by silhouette score.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

from src.data.preprocess import get_preprocessed_data
from src.models.clustering import (
    run_clustering,
    get_cluster_model,
    plot_elbow_and_silhouette,
    plot_clusters,
    create_cluster_profile,
)

CLUSTER_FIG_DIR = PROJECT_ROOT / "outputs" / "figures" / "clustering"
CLUSTER_FIG_DIR.mkdir(parents=True, exist_ok=True)

matplotlib.use("Agg")  # headless — save to file instead of screen
print("Imports OK | project root:", PROJECT_ROOT)

Imports OK | project root: /home/taqis/Pone/CSCI323_Project


In [2]:
data = get_preprocessed_data(str(PROJECT_ROOT / "data" / "raw" / "medical_insurance.csv"))

X_train = data["X_train"]
y_train = data["y_train"]

print("X_train shape:", X_train.shape)
print("Clustering uses columns:", ["age", "bmi"])
X_train[["age", "bmi"]].describe()

X_train shape: (1070, 9)
Clustering uses columns: ['age', 'bmi']


,age,bmi
count,1.070000e+03,1.070000e+03
mean,-1.992176e-16,1.776357e-16
std,1.000468e+00,1.000468e+00
min,-1.518194e+00,-2.410776e+00
25%,-8.784157e-01,-7.221574e-01
50%,1.016470e-02,-5.995415e-02
75%,8.276587e-01,6.519144e-01
max,1.751782e+00,3.731160e+00


In [3]:
# Elbow + silhouette curves to justify k selection
plot_elbow_and_silhouette(
    X_train,
    save_path=str(CLUSTER_FIG_DIR / "elbow_curve.png"),
)
print("Elbow and silhouette plots saved.")

Elbow and silhouette plots saved.


/home/taqis/Pone/CSCI323_Project/src/models/clustering.py:158: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/home/taqis/Pone/CSCI323_Project/src/models/clustering.py:170: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# Run clustering — k chosen automatically by silhouette score
result = run_clustering(X_train)

print(f"Chosen k       : {result['k']}")
print(f"Inertia        : {result['inertia']:.2f}")
print(f"Silhouette     : {result['silhouette_score']:.4f}")
print(f"Risk tiers     : {sorted(set(result['risk_tier']))}")

from collections import Counter
tier_counts = Counter(result["risk_tier"])
for tier, count in sorted(tier_counts.items()):
    print(f"  {tier:<14}: {count} customers ({count/len(result['risk_tier'])*100:.1f}%)")

Chosen k       : 2
Inertia        : 1302.05
Silhouette     : 0.3693
Risk tiers     : ['HIGH_RISK', 'LOW_RISK']
  HIGH_RISK     : 517 customers (48.3%)
  LOW_RISK      : 553 customers (51.7%)


In [5]:
# Scatter plot of clusters
plot_clusters(
    X_train,
    result,
    save_path=str(CLUSTER_FIG_DIR / "cluster_scatter.png"),
)
print("Cluster scatter plot saved.")

Cluster scatter plot saved.


/home/taqis/Pone/CSCI323_Project/src/models/clustering.py:198: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# Cluster profile table
profile = create_cluster_profile(X_train, result, expenses=y_train)
print("Cluster profile:")
print(profile.to_string(index=False))

Cluster profile:
 cluster       age       bmi risk_tier     expenses
       0  0.855846  0.324335 HIGH_RISK 16729.076035
       1 -0.800131 -0.303221  LOW_RISK 10183.334268


In [7]:
# Contract check
assert "labels"          in result
assert "risk_tier"       in result
assert "k"               in result
assert "inertia"         in result
assert "silhouette_score" in result
assert "cluster_centers" in result
assert set(result["risk_tier"]).issubset({"LOW_RISK", "MEDIUM_RISK", "HIGH_RISK"})
assert len(result["labels"]) == len(X_train)

print("All M2 contract checks passed.")

All M2 contract checks passed.
